In [1]:
import time
import math
import requests
from bs4 import BeautifulSoup
import pandas as pd
from collections import namedtuple
import re
#from sentence_transformers import SentenceTransformer, util
import time
from tqdm import tqdm
from collections import namedtuple
import random
import numpy as np


import json
import requests
from bs4 import BeautifulSoup

In [2]:
def limpia_precio(txt):
    if not txt: 
        return None
    # quita símbolos comunes de MercadoLibre (puntos de miles, comas, $)
    t = txt.replace("\xa0", " ").replace(".", "").replace(",", "").replace("$", "").strip()
    # a veces viene junto con centavos en otro span; nos quedamos con entero
    try:
        return int("".join(ch for ch in t if ch.isdigit()))
    except ValueError:
        return None
import re

def extrae_atributos(li):
    """
    Regresa dict con recamaras, banos, sup_m2, sup_construida_m2 (si aparece), etc.
    Toma los <li> dentro de ul.poly-attributes_list
    """
    attrs = {
        "recamaras": None,
        "banos": None,
        "superficie_m2": None,
        "superficie_construida_m2": None
    }

    items = li.select("ul.poly-attributes_list li.poly-attributes_list__item")
    textos = [it.get_text(" ", strip=True).lower() for it in items]

    for t in textos:
        # recámaras / habitaciones / dormitorios
        m = re.search(r"(\d+)\s*(recámaras|recamaras|habitaciones|dormitorios)", t)
        if m:
            attrs["recamaras"] = int(m.group(1))
            continue

        # baños
        m = re.search(r"(\d+)\s*(baños|banos)", t)
        if m:
            attrs["banos"] = int(m.group(1))
            continue

        # superficie construida
        m = re.search(r"(\d+(?:\.\d+)?)\s*m²\s*(construidos|construidas|construido|construida)", t)
        if m:
            attrs["superficie_construida_m2"] = float(m.group(1))
            continue

        # superficie (genérica) en m²
        m = re.search(r"(\d+(?:\.\d+)?)\s*m²", t)
        if m and attrs["superficie_m2"] is None:
            attrs["superficie_m2"] = float(m.group(1))
            continue

    return attrs

def parse_item(li):
    # Títulos
    nombre = None
    brand_span = li.select_one(".poly-component__brand")
    if brand_span:
        nombre = brand_span.get_text(strip=True)
        
    descripcion = None
    t = li.find("h2")
    if t:
        descripcion = t.get_text(strip=True)
    else:
        t = li.find("h3")
        if t: descripcion = t.get_text(strip=True)

    # Link
    a = li.find("a", href=True)
    link = a["href"] if a else None

    # Imagen
    img = li.find("img")
    imagen = img.get("data-src") or img.get("src") if img else None

    # Precios (ML usa varias clases; probamos alternativas)
    precio_actual = None
    precio_antes = None

    # Número entero del precio actual
    actual_span = li.select_one(".poly-price__current .andes-money-amount__fraction")
    if actual_span:
        precio_actual = limpia_precio(actual_span.get_text())

    # Posible precio anterior (tachado) aparece con otra clase o data-test-id
    antes = (li.select_one(".andes-money-amount.comparative-price .andes-money-amount__fraction")
             or li.select_one("[data-test-id='price-before'] .andes-money-amount__fraction")
             or li.select_one(".ui-search-price__second-line .price-tag__disabled .price-tag-amount"))
    if antes:
        precio_antes = limpia_precio(antes.get_text())
    # -------- ATRIBUTOS NUEVOS (recámaras, baños, superficie) --------
    recamaras = None
    banos = None
    superficie_m2 = None

    attrs = li.select("ul.poly-attributes_list li.poly-attributes_list__item")
    for it in attrs:
        txt = it.get_text(" ", strip=True).lower()

        m = re.search(r"(\d+)\s*(recámaras|recamaras|habitaciones|dormitorios)", txt)
        if m:
            recamaras = int(m.group(1))
            continue

        m = re.search(r"(\d+)\s*(baños|banos)", txt)
        if m:
            banos = int(m.group(1))
            continue

        m = re.search(r"(\d+(?:\.\d+)?)\s*m²", txt)
        if m:
            superficie_m2 = float(m.group(1))
            continue

    return Articulo(
        nombre,
        descripcion,
        recamaras,
        banos,
        superficie_m2,
        precio_actual,
        link,
        imagen
    )

    return Articulo(nombre, descripcion,  precio_actual, link, imagen)

Articulo = namedtuple("Articulo", ["nombre", "descripcion", "recamaras", "banos", "superficie_m2", "precio_actual", "link", "imagen"])
articulos = []

# ---------- Config ----------
PAIS = "com.mx"  # cambia a "com.co", "com.ar", etc.

PAGINAS = 1            # cuántas páginas raspar
ESPERA_S = 30         # pausa entre páginas (ser buen ciudadano)
BASE = f"https://listado.mercadolibre.{PAIS}"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "es-MX,es;q=0.9"
}

In [3]:
import requests
import re

# ------------------- Configuración básica ------------------- #

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "es-MX,es;q=0.9"
}

# ------------------- Función get_lat_lon ------------------- #

def get_lat_lon(url, headers=HEADERS):
    """
    Descarga la página del anuncio de MercadoLibre y busca
    'latitude' y 'longitude' en el HTML (normalmente vienen
    dentro de un JSON embebido en <script>).
    Devuelve (latitud, longitud) como floats o (None, None) si no las encuentra.
    """
    try:
        resp = requests.get(url, headers=headers, timeout=25)
        resp.raise_for_status()
    except requests.RequestException as e:
        print(f"[WARN] Error obteniendo la página: {e}")
        return None, None

    html = resp.text

    # Intento 1: patrón donde lat y lon están cerca en el mismo JSON
    # ... "latitude":19.12345,"longitude":-99.12345 ...
    m = re.search(
        r'"latitude"\s*:\s*"?([-0-9.]+)"?.*?"longitude"\s*:\s*"?([-0-9.]+)"?',
        html,
        flags=re.DOTALL
    )
    if m:
        try:
            lat = float(m.group(1))
            lon = float(m.group(2))
            return lat, lon
        except ValueError:
            pass

    # Intento 2: por separado (por si vienen en lugares distintos)
    lat_match = re.search(r'"latitude"\s*:\s*"?([-0-9.]+)"?', html)
    lon_match = re.search(r'"longitude"\s*:\s*"?([-0-9.]+)"?', html)

    if lat_match and lon_match:
        try:
            lat = float(lat_match.group(1))
            lon = float(lon_match.group(1))
            return lat, lon
        except ValueError:
            pass

    # Si llegamos aquí, no encontramos coordenadas
    print("[INFO] No se encontraron campos 'latitude'/'longitude' en el HTML.")
    return None, None

# ------------------- Uso con tu URL ------------------- #

url = "https://departamento.mercadolibre.com.mx/MLM-2433508491-departamento-en-venta-en-paseos-de-taxquena-coyoacan-_JM#polycard_client=search-nordic&search_layout=grid&position=1&type=item&tracking_id=53c09837-a3f1-45f4-bc99-b8ff7aba6b83"

lat, lon = get_lat_lon(url)
print("Latitud:", lat)
print("Longitud:", lon)

Latitud: 19.3475554
Longitud: -99.1251122


In [4]:
import requests
import re
import pandas as pd

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "es-MX,es;q=0.9"
}

def get_lat_lon(url, headers=HEADERS):
    """Devuelve solo latitud y longitud desde un anuncio de MercadoLibre."""
    try:
        resp = requests.get(url, headers=headers, timeout=25)
        resp.raise_for_status()
    except Exception as e:
        print(f"[WARN] Error al obtener la página: {e}")
        return None, None

    html = resp.text

    # Buscar lat y lon en el JSON interno
    m = re.search(
        r'"latitude"\s*:\s*([-0-9.]+).*?"longitude"\s*:\s*([-0-9.]+)',
        html,
        flags=re.DOTALL
    )
    if m:
        return float(m.group(1)), float(m.group(2))

    # Como fallback, buscar por separado
    lat_match = re.search(r'"latitude"\s*:\s*([-0-9.]+)', html)
    lon_match = re.search(r'"longitude"\s*:\s*([-0-9.]+)', html)

    if lat_match and lon_match:
        return float(lat_match.group(1)), float(lon_match.group(1))

    print("[INFO] No se encontraron coordenadas.")
    return None, None


# ---------------- USO ---------------- #

url = "https://departamento.mercadolibre.com.mx/MLM-2433508491-departamento-en-venta-en-paseos-de-taxquena-coyoacan-_JM#polycard_client=search-nordic&search_layout=grid&position=1&type=item&tracking_id=53c09837-a3f1-45f4-bc99-b8ff7aba6b83"

lat, lon = get_lat_lon(url)

# DataFrame con SOLO coordenadas
df = pd.DataFrame([{
    "latitud": lat,
    "longitud": lon
}])

df

,latitud,longitud
0,19.39068,-99.283699


In [5]:
import requests
from bs4 import BeautifulSoup

def get_descripcion(url, headers=HEADERS):
    """
    Devuelve la descripción completa del anuncio de MercadoLibre.
    Retorna None si no se encuentra.
    """
    try:
        resp = requests.get(url, headers=headers, timeout=25)
        resp.raise_for_status()
    except Exception as e:
        print(f"[WARN] Error al obtener la descripción: {e}")
        return None

    soup = BeautifulSoup(resp.text, "html.parser")

    # Primer intento: selector principal
    desc_node = soup.select_one("p.ui-pdp-description__content")

    # Segundo intento (por si cambia la clase)
    if not desc_node:
        desc_node = soup.select_one("[data-testid='content']")

    if not desc_node:
        print("[INFO] No se encontró la descripción en la página.")
        return None

    # Extraemos texto limpio
    descripcion = desc_node.get_text(separator=" ", strip=True)

    return descripcion

In [6]:
url_test = "https://departamento.mercadolibre.com.mx/MLM-2433508491-departamento-en-venta-en-paseos-de-taxquena-coyoacan-_JM"

descripcion = get_descripcion(url_test)
descripcion

'Departamento completamente remodelado en Paseos de Taxqueña, frente al Parque de las Montañas.\n\n- Recámara principal con vestidor y baño privado.\n- 2 recámaras secundarias con clóset, que comparten un baño completo.\n- Sala de TV para disfrutar de momentos de entretenimiento en familia.\n- Amplia sala y comedor con balcón, ideal para disfrutar de la vista y el aire libre.\n- Cocina equipada con moderna alacena de gran capacidad.\n- Cuarto de lavado independiente.\n- Baño de servicio adicional.\n- 2 lugares de estacionamiento techados para mayor comodidad (llegan a entrar 3 autos pequeños).\n- Elevador y vigilancia 24 horas en el edificio para tu seguridad.\n \nMantenimiento: $1,300 mensuales.\n\nEl departamento tiene una orientación al oriente en la recámara principal y sala/comedor, mientras que las dos recámaras secundarias están orientadas al poniente, lo que garantiza excelente iluminación natural durante el día. \n\nEs un espacio ideal para quienes buscan comodidad, modernidad

In [7]:
from datetime import date

hoy = date.today()
print(hoy)

2026-03-04


In [ ]:
# ---------- Scrape con paginado ----------
def scrap_page(PAGINAS, URL, HEADERS):
    articulos = []  # mejor local, para que no se quede sucio entre ejecuciones

    for page in range(1, PAGINAS + 1):
        sep = "" if URL.endswith("/") else "/"
        url_pag = URL if page == 1 else f"{URL}{sep}_Desde_{(page-1)*48+1}"

        try:
            resp = requests.get(url_pag, headers=HEADERS, timeout=20)
            resp.raise_for_status()
        except requests.RequestException as e:
            print(f"[WARN] {e} -> {url_pag}")
            break

        soup = BeautifulSoup(resp.text, "html.parser")

        # Cada item suele estar en <li> del layout de búsqueda
        items = soup.select("li.ui-search-layout__item, li.ui-search-layout__item--stack")
        if not items:
            items = soup.select("li.ui-search-result__wrapper")
        if not items:
            print(f"[WARN] No se hallaron items en página {page}.")
            break

        for li in items:
            try:
                art = parse_item(li)  # debe traer link, precio, baños, recámaras, etc.

                # Por si parse_item a veces regresa None
                if art is None:
                    continue

                # Convertimos a dict para poder añadir campos nuevos sin tocar el namedtuple
                row = art._asdict() if hasattr(art, "_asdict") else dict(art)

                lat, lon, desc = (None, None, None)

                link = row.get("link")
                if link:
                    # Coordenadas
                    try:
                        lat, lon = get_lat_lon(link, HEADERS)
                    except Exception as e:
                        print(f"[WARN] lat/lon error: {e}")

                    # Descripción
                    try:
                        desc = get_desc(link, HEADERS)
                    except Exception as e:
                        print(f"[WARN] desc error: {e}")

                    # Pausa entre anuncios (para no bloquear)
                    time.sleep(random.uniform(10, 30))

                row["latitud"] = lat
                row["longitud"] = lon
                row["descripcion"] = desc

                articulos.append(row)

            except Exception as exc:
                print(f"[WARN] item con error: {exc}")

        # Pausa entre páginas
        time.sleep(ESPERA_S)

    df = pd.DataFrame(articulos)
    return df

In [ ]:
URL_TEST = "https://inmuebles.mercadolibre.com.mx/departamentos/venta/distrito-federal/coyoacan/"

df_test = scrap_page(
    PAGINAS=1,
    URL=URL_TEST,
    HEADERS=HEADERS,
    ESPERA_S=2,          # pausa entre páginas
    espera_item=(3, 6)   # pausa entre anuncios (sube a (10,30) cuando confirmes que sirve)
)

df_test.head(5)